# **Archivo 3 — Comparación y clasificación SANOS vs ENFERMOS (cohorte completa, v3)**

**Qué hace este notebook.** Recorre las carpetas reales de la cohorte (`DATASET_NEUROIMAGEN/SANOS/<S#>` y `ENFERMOS/<E#>`), calcula **desde el `.tt.gz` de cada sujeto** las mismas métricas geométricas del Archivo 2 (conteo, longitud, tortuosidad, PCA λ₁–λ₃, elongación, relación de aspecto) para los 8 tractos témporo-límbicos (arcuato, uncinado, MdLF e ILF, izquierdo y derecho), las compara entre grupos y entrena clasificadores con **validación cruzada por sujeto**.

**Por qué así.** Ejecutar el Archivo 2 completo (gráficos 3D, TDI, ZIP) 76 veces no es necesario para comparar grupos: aquí se reutiliza solo su núcleo numérico, con la misma decodificación TinyTrack y la misma PCA. Las salidas de cada sujeto se guardan en caché, de modo que si Colab se interrumpe se retoma donde quedó.

**Resumen de decisiones de diseño**

| Decisión | Motivo |
|---|---|
| Etiquetas `sano` / `epilepsia` (no «control») | En el Archivo 2 `grupo_dx = "control"` designa a los pacientes (convención del investigador), lo contrario del uso habitual. Aquí se evita la ambigüedad. |
| El grupo sale de la **carpeta** (`SANOS`/`ENFERMOS`) y se comprueba contra la letra inicial del ID | Dos fuentes que deben coincidir; si no, el sujeto se marca. |
| Los nombres de los 8 clústeres se toman del `*_tract_stat*.txt` o del `*.tt.gz.txt` de **cada sujeto** | El orden de los clústeres no se asume igual en todos: se lee y se verifica con los conteos. |
| Modelo principal fijado de antemano: regresión logística L2 | Con 18 sanos, elegir «el mejor» entre varios modelos sobre la misma validación cruzada sobreestima el desempeño. Los otros modelos son análisis de sensibilidad. |
| Prueba de permutación y bootstrap de sujetos | Con n pequeño, el AUC puntual engaña; se reporta su incertidumbre. |
| Controles de sesgo | Se compara el protocolo de adquisición entre grupos y se prueba cuánto predicen solas variables que no son anatomía (p. ej. el total de streamlines). |

> **Alcance.** Es una comparación **transversal con 18 sanos y ≈58 pacientes**, sin covariables (edad, sexo, sitio) salvo que se aporte el archivo opcional `covariables.csv`. Un buen AUC aquí no demuestra un biomarcador clínico.

## **0. Cómo usarlo**

1. **Colab no lee la unidad `D:\`.** Copie la carpeta `DATASET_NEUROIMAGEN` a Google Drive (`MyDrive/DATASET_NEUROIMAGEN`) y ejecute en Colab; o ejecute en su PC con Jupyter local (la ruta de Windows ya está en la configuración).
2. Cada carpeta de sujeto debe contener un `*.tt.gz` (obligatorio) y, **para identificar los tractos**, al menos uno de: `*_tract_stat*.txt` (recomendado: trae además FA, MD, AD, RD, volumen) o `*.tt.gz.txt` (los 8 nombres de Autotrack, uno por línea). El `*.fz` no se usa aquí.
3. Ejecute **Ejecutar todo**. La extracción (Sección 2) tarda unos segundos por sujeto; los resultados quedan en `salida_archivo3/features/`.
4. Opcional: `covariables.csv` en la raíz del dataset, con columnas `paciente, edad, sexo` (y opcionalmente `sitio`) para comprobar el balance entre grupos.

In [ ]:
# ==============================================================================
# 0.1 CONFIGURACIÓN
# ==============================================================================
import os, re, io, gzip, sys, zlib, time, struct, glob, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import stats
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_DATASET = "/content/drive/MyDrive/DATASET_NEUROIMAGEN"      # <-- ajustar a donde lo suba en Drive
else:
    RAIZ_DATASET = r"D:\Maestria-Analitica-BI\08-Trabaja-Integrador\DATASET_NEUROIMAGEN"
RAIZ_DATASET = os.environ.get("RAIZ_DATASET", RAIZ_DATASET)

GRUPOS = {"SANOS": 0, "ENFERMOS": 1}          # carpeta -> etiqueta (1 = epilepsia = clase positiva)
NOMBRE_GRUPO = {0: "sano", 1: "epilepsia"}
PREFIJO_GRUPO = {"SANOS": "S", "ENFERMOS": "E"}
ESPERADO = {"SANOS": 18, "ENFERMOS": 58}      # tamaño de la cohorte según el inventario del investigador
EXCLUIR = set()                               # IDs a omitir a propósito (documente el motivo en la tesis)
VERIFICAR_INTEGRIDAD = True                   # lee cada .tt.gz completo para detectar archivos truncados
USAR_CACHE = True                             # reutiliza features ya calculadas (retoma si se interrumpe)
ASUMIR_ORDEN_AUTOTRACK = False                # True: si un sujeto no trae nombres, asume el orden de S1 (NO recomendado)
ORDEN_AUTOTRACK = ["arc_l", "arc_r", "unc_l", "unc_r", "mdlf_r", "mdlf_l", "ilf_r", "ilf_l"]   # orden observado en S1
ESCALA_TINYTRACK = 32.0                       # punto fijo del formato TinyTrack (igual que el Archivo 2)

SEMILLA = 42
N_PLIEGUES, N_REPETICIONES = 5, 5             # validación cruzada estratificada repetida (por sujeto)
N_PERMUTACIONES = 200
MIN_POR_CLASE = 8                             # por debajo de esto no se entrena ningún clasificador
UMBRAL_NAN = 0.30                             # se descartan variables con más de 30 % de valores ausentes

CARPETA_SALIDA_A3 = "./salida_archivo3/"
CARPETA_FEATURES = os.path.join(CARPETA_SALIDA_A3, "features")
os.makedirs(CARPETA_FEATURES, exist_ok=True)
print("Raíz del dataset:", RAIZ_DATASET, "| existe:", os.path.isdir(RAIZ_DATASET))

## **1. Inventario de la cohorte**

Se recorre `<raíz>/<grupo>/<paciente>/` y se anota, por sujeto, qué archivos hay. Un sujeto es **utilizable** si tiene un `.tt.gz` íntegro y alguna fuente de nombres de tractos (`tract_stat` o `.tt.gz.txt`). Se comprueba además que la letra inicial del ID (`S`/`E`) coincida con la carpeta y que no haya IDs repetidos. Todo lo que no cumple queda **listado con su motivo** (el comité de ética preguntará por los sujetos faltantes y excluidos).

In [ ]:
# ==============================================================================
# 1. INVENTARIO (basado en Archivo3_cargue_pacientes.py)
# ==============================================================================
def _clave_natural(nombre):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", nombre)]

def verificar_gzip(ruta):
    """True si el .gz se lee completo (sin truncar)."""
    try:
        with gzip.open(ruta, "rb") as f:
            while f.read(1 << 24):
                pass
        return True, ""
    except (EOFError, OSError, zlib.error) as e:
        return False, str(e)

def _buscar(carpeta, sufijo, contiene=None):
    out = []
    for f in os.listdir(carpeta):
        fl = f.lower()
        if fl.endswith(sufijo) and (contiene is None or contiene in fl):
            out.append(f)
    return sorted(out)

def construir_inventario(raiz=RAIZ_DATASET, grupos=GRUPOS, verificar=VERIFICAR_INTEGRIDAD):
    filas = []
    for grupo, etiqueta in grupos.items():
        carpeta_grupo = os.path.join(raiz, grupo)
        if not os.path.isdir(carpeta_grupo):
            print(f"⚠ No existe la carpeta del grupo: {carpeta_grupo}")
            continue
        pacientes = sorted((d for d in os.listdir(carpeta_grupo) if os.path.isdir(os.path.join(carpeta_grupo, d))),
                           key=_clave_natural)
        for pid in pacientes:
            if pid.upper() in EXCLUIR:
                continue
            carpeta = os.path.join(carpeta_grupo, pid)
            tts = _buscar(carpeta, ".tt.gz")
            fzs = _buscar(carpeta, ".fz")
            txts = _buscar(carpeta, ".tt.gz.txt")
            stats_ = _buscar(carpeta, ".txt", contiene="tract_stat")
            problemas = []
            if len(tts) == 0: problemas.append("falta_tt")
            if len(tts) > 1:  problemas.append(f"varios_tt({len(tts)})")
            if len(txts) > 1: problemas.append(f"varios_nombres({len(txts)})")
            if len(stats_) > 1: problemas.append(f"varios_tract_stat({len(stats_)})")
            ruta_tt = os.path.join(carpeta, tts[0]) if len(tts) == 1 else ""
            ruta_txt = os.path.join(carpeta, txts[0]) if len(txts) == 1 else ""
            ruta_stat = os.path.join(carpeta, stats_[0]) if len(stats_) == 1 else ""
            ruta_fz = os.path.join(carpeta, fzs[0]) if len(fzs) == 1 else ""
            if verificar and ruta_tt:
                ok, msg = verificar_gzip(ruta_tt)
                if not ok: problemas.append("tt_danado")
            if not ruta_txt and not ruta_stat and not ASUMIR_ORDEN_AUTOTRACK:
                problemas.append("sin_nombres_de_tractos")
            if not pid.upper().startswith(PREFIJO_GRUPO[grupo]):
                problemas.append(f"prefijo_no_coincide_con_carpeta_{grupo}")
            usable = bool(ruta_tt) and not any(p.startswith(("tt_danado", "sin_nombres", "prefijo", "varios_tt")) for p in problemas)
            filas.append({"paciente": pid.upper(), "grupo": grupo, "etiqueta": etiqueta,
                          "grupo_clinico": NOMBRE_GRUPO[etiqueta],
                          "ruta_tt": ruta_tt, "ruta_fz": ruta_fz, "ruta_nombres": ruta_txt, "ruta_stat": ruta_stat,
                          "tiene_fz": bool(ruta_fz), "tiene_nombres": bool(ruta_txt), "tiene_tract_stat": bool(ruta_stat),
                          "usable": usable, "problemas": ";".join(problemas)})
    return pd.DataFrame(filas)

inventario = construir_inventario()
if len(inventario) == 0:
    raise SystemExit("No se encontró ningún sujeto. Revise RAIZ_DATASET y la estructura <raíz>/SANOS|ENFERMOS/<ID>/.")

print(f"Sujetos encontrados: {len(inventario)}  |  utilizables: {int(inventario['usable'].sum())}")
resumen = inventario.groupby("grupo").agg(encontrados=("paciente", "count"), utilizables=("usable", "sum"),
                                          con_tract_stat=("tiene_tract_stat", "sum"), con_nombres_txt=("tiene_nombres", "sum"),
                                          con_fz=("tiene_fz", "sum"))
display(resumen)
for g, n in ESPERADO.items():
    n_real = int((inventario.grupo == g).sum())
    if n_real != n:
        print(f"AVISO: se esperaban {n} sujetos en {g} y hay {n_real}.")
obs = inventario[inventario.problemas != ""]
if len(obs):
    print("\nSujetos con observaciones:")
    display(obs[["grupo", "paciente", "usable", "problemas"]])
dup = inventario.paciente[inventario.paciente.duplicated(keep=False)].tolist()
if dup:
    print("⚠ IDs repetidos entre grupos:", dup)
inventario.to_csv(os.path.join(CARPETA_SALIDA_A3, "inventario_pacientes.csv"), index=False, encoding="utf-8-sig")

## **2. Extracción de características por sujeto (directamente del `.tt.gz`)**

Por cada sujeto utilizable se hace lo mismo que el Archivo 2, sin gráficos:

1. **Decodificación TinyTrack** (escala 1/32 de vóxel) y comprobación de que el número de streamlines coincide con el de etiquetas de clúster.
2. **Identificación de cada clúster** con los nombres del propio sujeto: primero `tract_stat` (cabecera de columnas) y, si no hay, el `.tt.gz.txt`. Si ambas fuentes existen y discrepan, o si el conteo de streamlines por clúster no coincide con `tract_stat`, el sujeto se **excluye y se lista**.
3. **Métricas por tracto**: número de streamlines (absoluto y relativo al total del sujeto), longitud media y mediana (mm), tortuosidad mediana, eigenvalores λ₁≥λ₂≥λ₃ de la covarianza de todos los puntos del clúster, elongación λ₁/λ₂ y relación de aspecto λ₁/λ₃. Si hay `tract_stat`, se añaden FA, MD, AD, RD, volumen, curl e irregularidad que calcula DSI Studio (la «elongation» de DSI Studio es **otra métrica**, se guarda como `elong_dsi` y no debe confundirse con λ₁/λ₂).
4. **Parámetros de adquisición** leídos del campo `report` del `.tt.gz` (direcciones, b-value, resolución), para detectar sesgo de protocolo entre grupos (Sección 3).

> **Espacio nativo.** Estas métricas no dependen del registro a MNI (son longitudes, razones y conteos), por lo que son comparables entre sujetos sin normalización espacial. Las coordenadas absolutas **no** se usan como variables.

In [ ]:
# ==============================================================================
# 2.1 FUNCIONES DE LECTURA, DECODIFICACIÓN Y MÉTRICAS
# ==============================================================================
CODIGOS = ["arc", "unc", "mdlf", "ilf"]

def cargar_dsi(ruta):
    with gzip.open(ruta, "rb") as f:
        return sio.loadmat(io.BytesIO(f.read()))

def decodificar_tinytrack(track_bytes, escala=ESCALA_TINYTRACK):
    """Igual que el Archivo 2: cada registro = uint32 (3·n_puntos), 3 int32 (primer punto) y deltas int8."""
    out, n, pos = [], len(track_bytes), 0
    while pos < n:
        count = struct.unpack_from("<I", track_bytes, pos)[0]
        x0, y0, z0 = struct.unpack_from("<iii", track_bytes, pos + 4)
        n_pts = count // 3
        pts = np.empty((n_pts, 3), dtype=np.int64)
        pts[0] = (x0, y0, z0)
        if n_pts > 1:
            d = np.frombuffer(track_bytes, dtype=np.int8, count=count - 3, offset=pos + 16).astype(np.int64).reshape(n_pts - 1, 3)
            pts[1:] = np.cumsum(d, axis=0) + pts[0]
        out.append(pts / escala)
        pos += 13 + count
    return out

def _norm_nombre(n):
    """'s12_arc_l' -> ('arc','l'); flm -> mdlf; fli -> ilf. None si no se reconoce."""
    m = re.search(r"(?:^|[_\-\s.])(arc|unc|flm|fli|mdlf|ilf)[_\-]([lr])(?:$|[_\-\s.])", str(n).strip().lower())
    if not m:
        return None
    return {"flm": "mdlf", "fli": "ilf"}.get(m.group(1), m.group(1)), m.group(2)

def leer_nombres_txt(ruta):
    with open(ruta, encoding="utf-8", errors="replace") as f:
        return [l.strip() for l in f if l.strip()]

CLAVES_STAT = {"number of tracts": "n_stat", "fa": "fa", "md": "md", "ad": "ad", "rd": "rd",
               "total volume(mm^3)": "volumen", "curl": "curl", "irregularity": "irregularidad",
               "elongation": "elong_dsi", "mean length(mm)": "long_media_stat"}

def leer_tract_stat(ruta):
    """Devuelve (nombres_en_orden, {metrica: [valores por tracto]}) del reporte tract_stat de DSI Studio."""
    with open(ruta, encoding="utf-8", errors="replace") as f:
        filas = [l.rstrip("\n").split("\t") for l in f if l.strip()]
    nombres = [h.strip() for h in filas[0][1:] if h.strip()]
    datos = {}
    for r in filas[1:]:
        clave = CLAVES_STAT.get(r[0].strip().lower())
        if clave is None:
            continue
        vals = []
        for v in r[1:1 + len(nombres)]:
            try: vals.append(float(v))
            except ValueError: vals.append(np.nan)
        if len(vals) == len(nombres):
            datos[clave] = vals
    return nombres, datos

def resolver_mapeo(fila, ids_cluster):
    """cluster_id -> (codigo, lado). Prioridad: tract_stat > .tt.gz.txt. Devuelve (mapeo, fuente, stat, problemas)."""
    problemas, stat, candidatos = [], None, {}
    if fila.ruta_stat:
        nom, datos = leer_tract_stat(fila.ruta_stat)
        candidatos["tract_stat"] = nom; stat = datos
    if fila.ruta_nombres:
        candidatos["txt"] = leer_nombres_txt(fila.ruta_nombres)
    if not candidatos:
        if ASUMIR_ORDEN_AUTOTRACK:
            candidatos["orden_asumido"] = ORDEN_AUTOTRACK
            problemas.append("orden_asumido_no_verificado")
        else:
            return None, "", None, ["sin_nombres_de_tractos"]
    parseados = {k: [_norm_nombre(n) for n in v] for k, v in candidatos.items()}
    if len(parseados) > 1:
        a, b = list(parseados.values())[:2]
        if a != b:
            problemas.append("nombres_tract_stat_y_txt_discrepan")
    fuente = "tract_stat" if "tract_stat" in parseados else list(parseados)[0]
    lista = parseados[fuente]
    ids = sorted(int(i) for i in np.unique(ids_cluster))
    if len(lista) != len(ids) or any(x is None for x in lista):
        problemas.append(f"nombres_no_reconocidos(n_nombres={len(lista)},n_clusters={len(ids)})")
        return None, fuente, stat, problemas
    if len(set(lista)) != 8 or {c for c, _ in lista} != set(CODIGOS):
        problemas.append("mapeo_no_cubre_los_8_tractos")
        return None, fuente, stat, problemas
    return {cid: lista[i] for i, cid in enumerate(ids)}, fuente, stat, problemas

def metricas_por_cluster(sl, cl, ref):
    """Longitud, tortuosidad y PCA por clúster. La covarianza se obtiene de estadísticos suficientes (equivale a np.cov)."""
    n = len(sl)
    long_ = np.empty(n); dext = np.empty(n); npts = np.empty(n); S1 = np.empty((n, 3)); S2 = np.empty((n, 9))
    for i, p in enumerate(sl):
        long_[i] = np.sqrt((np.diff(p, axis=0) ** 2).sum(axis=1)).sum()
        dext[i] = np.linalg.norm(p[-1] - p[0])
        q = p - ref
        npts[i] = len(q); S1[i] = q.sum(axis=0); S2[i] = (q.T @ q).ravel()
    tort = np.where(dext > 1e-6, long_ / np.maximum(dext, 1e-6), np.nan)
    res = {}
    for cid in np.unique(cl):
        m = cl == cid
        N = npts[m].sum(); s1 = S1[m].sum(axis=0); s2 = S2[m].sum(axis=0).reshape(3, 3)
        mu = s1 / N
        cov = (s2 - N * np.outer(mu, mu)) / (N - 1)
        ev = np.sort(np.linalg.eigvalsh(cov))[::-1]
        res[int(cid)] = dict(n_streamlines=int(m.sum()), long_media=float(long_[m].mean()), long_mediana=float(np.median(long_[m])),
                             tort_mediana=float(np.nanmedian(tort[m])),
                             lambda1=float(ev[0]), lambda2=float(ev[1]), lambda3=float(ev[2]),
                             elongacion=float(ev[0] / ev[1]) if ev[1] > 1e-9 else np.nan,
                             aspecto=float(ev[0] / ev[2]) if ev[2] > 1e-9 else np.nan)
    return res

def parametros_adquisicion(tt):
    """Lee del campo 'report' del .tt.gz los parámetros de adquisición (para detectar sesgo de protocolo)."""
    try:
        rep = bytes(np.asarray(tt["report"]).astype(np.uint8).ravel()).decode("utf-8", errors="replace")
    except Exception:
        rep = ""
    def _f(pat):
        m = re.search(pat, rep)
        return float(m.group(1)) if m else np.nan
    esquema = re.search(r"A (\w+) diffusion scheme", rep)
    return {"acq_esquema": esquema.group(1) if esquema else "",
            "acq_direcciones": _f(r"total of (\d+) diffusion sampling directions"),
            "acq_bvalue": _f(r"b-value was ([\d.]+)"),
            "acq_res_plano_mm": _f(r"in-plane resolution was ([\d.]+)"),
            "acq_espesor_mm": _f(r"slice thickness was ([\d.]+)"),
            "acq_dim": "x".join(str(int(v)) for v in np.asarray(tt["dimension"]).ravel()) if "dimension" in tt else ""}

def extraer_sujeto(fila):
    """Devuelve (fila_ancha_dict, problemas). Lanza excepción si el .tt.gz no se puede decodificar."""
    tt = cargar_dsi(fila.ruta_tt)
    cl = np.asarray(tt["cluster"]).ravel().astype(int)
    sl = decodificar_tinytrack(np.asarray(tt["track"]).astype(np.uint8).tobytes())
    if len(sl) != len(cl):
        raise ValueError(f"descuadre streamlines/clusters: {len(sl)} vs {len(cl)}")
    mapeo, fuente, stat, problemas = resolver_mapeo(fila, cl)
    if mapeo is None:
        return None, problemas
    ref = np.asarray(tt["dimension"]).ravel().astype(float) / 2.0
    met = metricas_por_cluster(sl, cl, ref)
    ids = sorted(mapeo)
    fila_out = {"paciente": fila.paciente, "grupo": fila.grupo, "etiqueta": int(fila.etiqueta), "grupo_clinico": fila.grupo_clinico,
                "fuente_nombres": fuente, "n_total_streamlines": int(len(cl))}
    fila_out.update(parametros_adquisicion(tt))
    for i, cid in enumerate(ids):
        cod, lado = mapeo[cid]
        pref = f"{cod}_{lado}__"
        m = met[cid]
        for k, v in m.items():
            fila_out[pref + k] = v
        fila_out[pref + "n_rel"] = m["n_streamlines"] / max(len(cl), 1)
        if stat is not None:
            for k, vals in stat.items():
                if k != "n_stat":
                    fila_out[pref + k] = vals[i]
            if "n_stat" in stat and int(round(stat["n_stat"][i])) != m["n_streamlines"]:
                problemas.append(f"conteo_no_coincide_con_tract_stat({cod}_{lado}: {m['n_streamlines']} vs {int(round(stat['n_stat'][i]))})")
    return fila_out, problemas

print("Funciones de extracción definidas.")

In [ ]:
# ==============================================================================
# 2.2 EJECUCIÓN POR LOTES (con caché; un sujeto con error no detiene el lote)
# ==============================================================================
filas_ok, estado = [], []
t_ini = time.time()
utilizables = inventario[inventario.usable].reset_index(drop=True)
for k, fila in enumerate(utilizables.itertuples(), 1):
    ruta_cache = os.path.join(CARPETA_FEATURES, f"{fila.grupo}_{fila.paciente}.csv")
    if USAR_CACHE and os.path.exists(ruta_cache):
        d = pd.read_csv(ruta_cache).iloc[0].to_dict()
        filas_ok.append(d); estado.append((fila.paciente, fila.grupo, "ok (caché)", "")); continue
    t0 = time.time()
    try:
        d, probs = extraer_sujeto(fila)
    except Exception as e:
        estado.append((fila.paciente, fila.grupo, "error", f"{type(e).__name__}: {e}"))
        print(f"[{k}/{len(utilizables)}] ✗ {fila.grupo}/{fila.paciente}: {type(e).__name__}: {e}")
        continue
    if d is None or any(p.startswith(("conteo_no_coincide", "nombres_tract_stat_y_txt_discrepan")) for p in probs):
        estado.append((fila.paciente, fila.grupo, "excluido", ";".join(probs)))
        print(f"[{k}/{len(utilizables)}] ✗ {fila.grupo}/{fila.paciente} excluido: {';'.join(probs)}")
        continue
    d["avisos"] = ";".join(probs)
    pd.DataFrame([d]).to_csv(ruta_cache, index=False, encoding="utf-8-sig")
    filas_ok.append(d); estado.append((fila.paciente, fila.grupo, "ok", ";".join(probs)))
    print(f"[{k}/{len(utilizables)}] ✓ {fila.grupo}/{fila.paciente} ({time.time() - t0:.0f} s)")

df_estado = pd.DataFrame(estado, columns=["paciente", "grupo", "estado", "detalle"])
df_feat = pd.DataFrame(filas_ok)
print(f"\nExtracción terminada en {(time.time() - t_ini) / 60:.1f} min.")
display(df_estado.groupby(["grupo", "estado"]).size().unstack(fill_value=0))
malos = df_estado[~df_estado.estado.str.startswith("ok")]
if len(malos):
    print("Sujetos NO incluidos en el análisis (con motivo):"); display(malos)
df_feat.to_csv(os.path.join(CARPETA_SALIDA_A3, "features_por_sujeto_a3.csv"), index=False, encoding="utf-8-sig")

## **3. Control de calidad y sesgos antes de comparar**

**3.1 Cohorte efectiva.** Cuántos sujetos de cada grupo entran realmente al análisis.

**3.2 Protocolo de adquisición por grupo.** Si sanos y pacientes se escanearon con protocolos distintos (direcciones, b-value, resolución), un clasificador puede aprender el protocolo y no la enfermedad. Se compara cada parámetro entre grupos con los datos del propio `.tt.gz`.

**3.3 Covariables (opcional).** Si existe `covariables.csv` (`paciente, edad, sexo`), se comprueba el balance de edad y sexo entre grupos.

In [ ]:
# ==============================================================================
# 3. CONTROL DE CALIDAD Y SESGOS
# ==============================================================================
if len(df_feat) == 0:
    raise SystemExit("No hay sujetos con características extraídas; revise la Sección 2.")
df_feat["etiqueta"] = df_feat["etiqueta"].astype(int)
n_por_clase = df_feat.etiqueta.value_counts().reindex([0, 1], fill_value=0)
print("Cohorte efectiva ->  sanos:", int(n_por_clase[0]), "| epilepsia:", int(n_por_clase[1]),
      f"| proporción de epilepsia: {n_por_clase[1] / n_por_clase.sum():.0%}")
HAY_AMBAS = n_por_clase.min() >= 1
PUEDE_MODELAR = n_por_clase.min() >= MIN_POR_CLASE
if not PUEDE_MODELAR:
    print(f"⚠ Menos de {MIN_POR_CLASE} sujetos en alguna clase: se muestran descriptivos, pero NO se entrenan clasificadores.")

# 3.2 Adquisición por grupo
COLS_ACQ = ["acq_direcciones", "acq_bvalue", "acq_res_plano_mm", "acq_espesor_mm"]
filas_acq = []
for c in COLS_ACQ + ["n_total_streamlines"]:
    a = df_feat.loc[df_feat.etiqueta == 0, c].dropna(); b = df_feat.loc[df_feat.etiqueta == 1, c].dropna()
    if len(a) and len(b):
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue if (a.nunique() + b.nunique() > 2) else np.nan
        filas_acq.append({"variable": c, "sano_mediana": a.median(), "sano_rango": f"{a.min():g}–{a.max():g}",
                          "epilepsia_mediana": b.median(), "epilepsia_rango": f"{b.min():g}–{b.max():g}", "p_MannWhitney": p,
                          "valores_distintos": int(pd.concat([a, b]).nunique())})
df_acq = pd.DataFrame(filas_acq)
display(df_acq.round(4))
df_acq.to_csv(os.path.join(CARPETA_SALIDA_A3, "adquisicion_por_grupo.csv"), index=False, encoding="utf-8-sig")
distintos = df_acq[df_acq.variable.isin(COLS_ACQ) & (df_acq.valores_distintos > 1)]
if len(distintos):
    print("⚠ Hay parámetros de adquisición que varían entre sujetos:", list(distintos.variable),
          "-> revise si coinciden con el grupo (posible sesgo de protocolo).")
else:
    print("✓ Los parámetros de adquisición leídos del .tt.gz son idénticos en toda la cohorte.")

# 3.3 Covariables opcionales
RUTA_COV = os.path.join(RAIZ_DATASET, "covariables.csv")
if os.path.exists(RUTA_COV):
    cov = pd.read_csv(RUTA_COV); cov["paciente"] = cov["paciente"].astype(str).str.upper()
    m = df_feat[["paciente", "etiqueta"]].merge(cov, on="paciente", how="left")
    print(f"\nCovariables disponibles para {m.edad.notna().sum() if 'edad' in m else 0}/{len(m)} sujetos.")
    if "edad" in m and m.edad.notna().sum() > 4:
        a, b = m.loc[m.etiqueta == 0, "edad"].dropna(), m.loc[m.etiqueta == 1, "edad"].dropna()
        print(f"Edad  sanos {a.median():.1f} [{a.min():.0f}–{a.max():.0f}] | epilepsia {b.median():.1f} [{b.min():.0f}–{b.max():.0f}] | "
              f"p (Mann-Whitney) = {stats.mannwhitneyu(a, b).pvalue:.3f}")
    if "sexo" in m and m.sexo.notna().sum() > 4:
        tab = pd.crosstab(m.etiqueta, m.sexo); display(tab)
        if tab.shape == (2, 2): print(f"Sexo: p (Fisher) = {stats.fisher_exact(tab.values)[1]:.3f}")
else:
    print("\n(No hay covariables.csv: el balance de edad y sexo entre grupos NO se pudo comprobar; es una limitación.)")

## **4. Comparación univariada SANO vs EPILEPSIA**

Para cada variable de cada tracto se compara el grupo epilepsia contra el sano con la **prueba U de Mann-Whitney** (no asume normalidad) y se reporta el **tamaño de efecto δ de Cliff** (−1 a 1; positivo = valores mayores en epilepsia). Como se prueban muchas variables a la vez, los valores p se corrigen por **Holm** (control del error familiar) y por **Benjamini-Hochberg** (tasa de falsos descubrimientos, *q*). Además se calcula el **índice de lateralidad** LI = (I − D)/(I + D) para cada tracto bilateral.

> Con 18 sanos, la potencia es limitada: un efecto que no sale significativo **no** demuestra ausencia de diferencia, y un efecto significativo sin corrección puede ser casualidad. Se debe mirar el tamaño de efecto y la q, no el p sin corregir.

In [ ]:
# ==============================================================================
# 4. COMPARACIÓN UNIVARIADA
# ==============================================================================
def cliff_delta(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    u = stats.mannwhitneyu(a, b, alternative="two-sided", method="auto").statistic
    return 2 * u / (len(a) * len(b)) - 1

def ajuste_holm(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    aj = np.maximum.accumulate((n - np.arange(n)) * p[o]); out = np.empty(n); out[o] = np.minimum(aj, 1.0); return out

def ajuste_bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)[::-1]
    aj = np.minimum.accumulate(p[o] * n / (n - np.arange(n))); out = np.empty(n); out[o[::-1]] = np.minimum(aj[::-1], 1.0); return out

# índices de lateralidad (I - D)/(I + D) para variables positivas
def agregar_lateralidad(df):
    df = df.copy()
    for cod in CODIGOS:
        for var in ["n_rel", "long_media", "elongacion", "aspecto", "fa"]:
            l, r = f"{cod}_l__{var}", f"{cod}_r__{var}"
            if l in df and r in df:
                den = df[l] + df[r]
                df[f"LI__{cod}__{var}"] = np.where(den > 0, (df[l] - df[r]) / den, np.nan)
    return df
df_feat = agregar_lateralidad(df_feat)

COLS_TRACTO = [c for c in df_feat.columns if "__" in c and not c.startswith("LI__")]
COLS_LI = [c for c in df_feat.columns if c.startswith("LI__")]
COLS_MODELO = COLS_TRACTO + COLS_LI
# descartar variables con demasiados ausentes o constantes
COLS_MODELO = [c for c in COLS_MODELO if df_feat[c].isna().mean() <= UMBRAL_NAN and df_feat[c].nunique(dropna=True) > 1]
print(f"Variables candidatas: {len(COLS_MODELO)} ({sum(c in COLS_TRACTO for c in COLS_MODELO)} por tracto + {sum(c in COLS_LI for c in COLS_MODELO)} de lateralidad)")

if HAY_AMBAS and n_por_clase.min() >= 3:
    filas = []
    for c in COLS_MODELO:
        a = df_feat.loc[df_feat.etiqueta == 0, c].dropna(); b = df_feat.loc[df_feat.etiqueta == 1, c].dropna()
        if len(a) < 3 or len(b) < 3: continue
        p = stats.mannwhitneyu(b, a, alternative="two-sided").pvalue
        filas.append({"variable": c, "mediana_sano": a.median(), "mediana_epilepsia": b.median(),
                      "delta_Cliff": cliff_delta(b, a), "p": p, "n_sano": len(a), "n_epilepsia": len(b)})
    df_uni = pd.DataFrame(filas)
    df_uni["p_Holm"] = ajuste_holm(df_uni.p); df_uni["q_BH"] = ajuste_bh(df_uni.p)
    df_uni = df_uni.sort_values("p").reset_index(drop=True)
    df_uni.to_csv(os.path.join(CARPETA_SALIDA_A3, "comparacion_univariada.csv"), index=False, encoding="utf-8-sig")
    print(f"{len(df_uni)} variables comparadas | p<0.05 sin corregir: {(df_uni.p < .05).sum()} | q_BH<0.05: {(df_uni.q_BH < .05).sum()} | p_Holm<0.05: {(df_uni.p_Holm < .05).sum()}")
    print(f"(Por azar, con {len(df_uni)} pruebas se esperarían ≈{0.05 * len(df_uni):.1f} p<0.05 aunque no existiera ninguna diferencia.)")
    display(df_uni.head(15).round(4))

    top = df_uni.head(6).variable.tolist()
    fig, axs = plt.subplots(2, 3, figsize=(12, 6.5)); rng = np.random.default_rng(SEMILLA)
    for ax, c in zip(axs.ravel(), top):
        datos = [df_feat.loc[df_feat.etiqueta == g, c].dropna().values for g in (0, 1)]
        ax.boxplot(datos, widths=0.5, showfliers=False, tick_labels=["sano", "epilepsia"])
        for i, d in enumerate(datos, 1):
            ax.scatter(i + rng.uniform(-0.12, 0.12, len(d)), d, s=14, alpha=0.6, color="#2c3e50")
        ax.set_title(c.replace("__", " · "), fontsize=9)
    plt.suptitle("Seis variables con menor p (sin corregir) — cada punto es un sujeto", fontsize=11)
    plt.tight_layout(); plt.savefig(CARPETA_SALIDA_A3 + "top_variables_univariado.png", dpi=150, bbox_inches="tight"); plt.show()
else:
    df_uni = pd.DataFrame(); print("Se necesitan al menos 3 sujetos por grupo para la comparación univariada.")

## **5. Clasificación SANO vs EPILEPSIA con validación cruzada por sujeto**

**Unidad de análisis = el sujeto** (una fila por persona), así que ninguna persona aparece a la vez en entrenamiento y prueba. La validación cruzada es **estratificada** (cada pliegue conserva la proporción 18 : 58) y se **repite** con distintas particiones para medir su variabilidad. Imputación (mediana) y estandarización se ajustan **solo con el pliegue de entrenamiento** dentro de un *pipeline*, para evitar fuga de información.

* **Modelo principal (fijado de antemano):** regresión logística L2 con pesos de clase balanceados.
* **Sensibilidad:** *random forest*, SVM con núcleo RBF y *gradient boosting* (mismas particiones). No se elige «el mejor»: elegir el máximo entre varios modelos sobre la misma validación cruzada la infla.
* **Referencias que el modelo debe superar:** (i) azar/mayoritaria (AUC 0.5) y (ii) un modelo que usa **solo el total de streamlines** y los parámetros de adquisición: si esto ya predice bien, la señal no es anatomía de los tractos.
* **Métricas:** AUC, exactitud balanceada, MCC, sensibilidad (detectar epilepsia) y especificidad (reconocer sano). Con 76 % de pacientes, la exactitud simple no sirve: predecir «epilepsia» siempre da 76 %.
* **Incertidumbre:** IC95 % del AUC por *bootstrap* de sujetos y **prueba de permutación** (¿el AUC supera al que se obtiene con etiquetas barajadas?).

In [ ]:
# ==============================================================================
# 5. CLASIFICACIÓN (validación cruzada estratificada, repetida, por sujeto)
# ==============================================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, matthews_corrcoef, confusion_matrix, roc_curve

def _pipe(clf): return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clf)
FABRICAS = {
    "Regresión logística L2 (principal)": lambda: _pipe(LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)),
    "Random forest": lambda: _pipe(RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", min_samples_leaf=2, random_state=SEMILLA, n_jobs=-1)),
    "SVM (RBF)": lambda: _pipe(SVC(C=1.0, kernel="rbf", class_weight="balanced", probability=True, random_state=SEMILLA)),
    "Gradient boosting": lambda: _pipe(HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=100, class_weight="balanced", random_state=SEMILLA)),
}

def metricas(y, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"AUC": roc_auc_score(y, p), "ExactitudBalanceada": balanced_accuracy_score(y, pred), "MCC": matthews_corrcoef(y, pred),
            "Sensibilidad": tp / max(tp + fn, 1), "Especificidad": tn / max(tn + fp, 1)}

def cv_oof(factory, X, y, n_pl, n_rep, semilla=SEMILLA, coefs=None):
    """Predicciones fuera de pliegue: matriz (n_rep, n). Cada sujeto se predice una vez por repetición con un modelo que no lo vio."""
    oof = np.full((n_rep, len(y)), np.nan)
    for r in range(n_rep):
        for tr, te in StratifiedKFold(n_pl, shuffle=True, random_state=semilla + r).split(X, y):
            m = factory(); m.fit(X[tr], y[tr]); oof[r, te] = m.predict_proba(X[te])[:, 1]
            if coefs is not None and r == 0:
                coefs.append(m[-1].coef_[0].copy())
    return oof

def resumen_cv(oof, y):
    por_rep = pd.DataFrame([metricas(y, oof[r]) for r in range(len(oof))])
    return por_rep.mean(), por_rep.std()

def ic_auc_bootstrap(y, p, B=2000, semilla=SEMILLA):
    rng = np.random.default_rng(semilla); n = len(y); aucs = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(set(y[i])) == 2: aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

if PUEDE_MODELAR:
    y = df_feat.etiqueta.values.astype(int)
    X = df_feat[COLS_MODELO].values.astype(float)
    n_pl = int(max(2, min(N_PLIEGUES, n_por_clase.min())))
    print(f"X: {X.shape[0]} sujetos × {X.shape[1]} variables | {n_pl} pliegues × {N_REPETICIONES} repeticiones")
    resultados, oofs, coefs_lr = [], {}, []
    for nombre, fab in FABRICAS.items():
        t0 = time.time()
        oof = cv_oof(fab, X, y, n_pl, N_REPETICIONES, coefs=coefs_lr if "principal" in nombre else None)
        oofs[nombre] = oof; media, sd = resumen_cv(oof, y); ic = ic_auc_bootstrap(y, np.nanmean(oof, axis=0))
        resultados.append({"modelo": nombre, **{k: media[k] for k in media.index}, "AUC_sd_repeticiones": sd["AUC"],
                           "AUC_IC95": f"[{ic[0]:.2f}, {ic[1]:.2f}]"})
        print(f"  {nombre}: AUC {media['AUC']:.3f}  ({time.time() - t0:.0f} s)")
    # referencias (no anatómicas)
    cols_ref = [c for c in ["n_total_streamlines"] + COLS_ACQ if c in df_feat and df_feat[c].nunique(dropna=True) > 1]
    if cols_ref:
        Xr = df_feat[cols_ref].values.astype(float)
        oof_r = cv_oof(FABRICAS["Regresión logística L2 (principal)"], Xr, y, n_pl, N_REPETICIONES)
        media, sd = resumen_cv(oof_r, y); ic = ic_auc_bootstrap(y, np.nanmean(oof_r, axis=0))
        resultados.append({"modelo": "REFERENCIA no anatómica: total de streamlines + adquisición", **{k: media[k] for k in media.index},
                           "AUC_sd_repeticiones": sd["AUC"], "AUC_IC95": f"[{ic[0]:.2f}, {ic[1]:.2f}]"})
    resultados.append({"modelo": "REFERENCIA azar / siempre «epilepsia»", "AUC": 0.5, "ExactitudBalanceada": 0.5, "MCC": 0.0,
                       "Sensibilidad": 1.0, "Especificidad": 0.0, "AUC_sd_repeticiones": np.nan, "AUC_IC95": "—"})
    df_res = pd.DataFrame(resultados)
    display(df_res.round(3))
    df_res.to_csv(os.path.join(CARPETA_SALIDA_A3, "resultados_modelos.csv"), index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for nombre, oof in oofs.items():
        p = np.nanmean(oof, axis=0); fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, lw=2.2 if "principal" in nombre else 1.2, label=f"{nombre.split(' (')[0]} (AUC {roc_auc_score(y, p):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("1 − especificidad"); ax.set_ylabel("Sensibilidad")
    ax.set_title("ROC fuera de pliegue (CV por sujeto, promedio de repeticiones)"); ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout(); plt.savefig(CARPETA_SALIDA_A3 + "roc_modelos.png", dpi=200); plt.show()
else:
    print("No se entrenan clasificadores (clases insuficientes).")

In [ ]:
# ==============================================================================
# 5.1 PRUEBA DE PERMUTACIÓN DEL MODELO PRINCIPAL
# ==============================================================================
if PUEDE_MODELAR:
    fab = FABRICAS["Regresión logística L2 (principal)"]
    auc_obs = roc_auc_score(y, np.nanmean(oofs["Regresión logística L2 (principal)"], axis=0))
    rng = np.random.default_rng(SEMILLA); nulo = []
    t0 = time.time()
    for i in range(N_PERMUTACIONES):
        yp = rng.permutation(y)
        o = cv_oof(fab, X, yp, n_pl, 1, semilla=SEMILLA + 1000 + i)
        nulo.append(roc_auc_score(yp, o[0]))
    nulo = np.array(nulo); pval = (1 + (nulo >= auc_obs).sum()) / (1 + len(nulo))
    print(f"AUC observado (promedio de repeticiones): {auc_obs:.3f} | AUC con etiquetas barajadas: media {nulo.mean():.3f}, "
          f"percentil 95 = {np.percentile(nulo, 95):.3f} | p de permutación = {pval:.4f} ({N_PERMUTACIONES} permutaciones, {time.time() - t0:.0f} s)")
    print("Lectura: el p de permutación indica si el AUC supera al azar; NO indica que el modelo sirva clínicamente ni que generalice a otro hospital.")
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); ax.hist(nulo, bins=20, color="#bdc3c7"); ax.axvline(auc_obs, color="#c0392b", lw=2, label=f"observado {auc_obs:.2f}")
    ax.set_xlabel("AUC con etiquetas barajadas"); ax.set_ylabel("Frecuencia"); ax.legend(); plt.tight_layout()
    plt.savefig(CARPETA_SALIDA_A3 + "permutacion_auc.png", dpi=200); plt.show()

### **5.2 ¿Qué variables usa el modelo principal?**

Coeficientes estandarizados de la regresión logística en cada pliegue de la primera repetición. Se muestra el coeficiente medio y en qué fracción de pliegues conserva el mismo signo (estabilidad). **Es una descripción del modelo, no una prueba de que ese tracto esté alterado.** Coeficientes positivos empujan hacia «epilepsia».

In [ ]:
if PUEDE_MODELAR and coefs_lr:
    C = np.array(coefs_lr)
    df_coef = pd.DataFrame({"variable": COLS_MODELO, "coef_medio": C.mean(axis=0),
                            "estabilidad_signo": np.maximum((C > 0).mean(axis=0), (C < 0).mean(axis=0))})
    df_coef["abs"] = df_coef.coef_medio.abs(); df_coef = df_coef.sort_values("abs", ascending=False).drop(columns="abs").reset_index(drop=True)
    df_coef.to_csv(os.path.join(CARPETA_SALIDA_A3, "coeficientes_modelo_principal.csv"), index=False, encoding="utf-8-sig")
    display(df_coef.head(15).round(3))

## **6. Alcance de este notebook y relación con la CNN**

* Este notebook compara grupos y clasifica con **métricas por tracto**. Es el análisis principal defendible con la cohorte actual.
* La **CNN** (2D sobre cortes TDI o 3D sobre el tensor de 8 canales) es un complemento exploratorio: necesita ejecutar el Archivo 2 por sujeto (TDI y tensor) y, para la versión 3D, un registro a un espacio común cuya calidad debe revisarse en un visor. Con 18 sanos una CNN se sobreajusta con facilidad; si se usa, debe compararse contra los resultados de la Sección 5 con **la misma validación por sujeto**, y solo es útil si los supera con claridad.
* El Archivo 2 sigue siendo el análisis detallado **por sujeto** (geometría, lateralidad, validación de reglas).

## **7. Limitaciones**

1. **Muestra pequeña y desbalanceada** (18 sanos frente a ≈58 pacientes): los intervalos son amplios y la especificidad es la métrica más inestable.
2. **Sin validación externa:** la validación cruzada es interna; no equivale a probar el modelo en otro hospital o escáner.
3. **Covariables:** edad y sexo (y sitio) pueden confundir; solo se comprueban si se aporta `covariables.csv`. Los parámetros de adquisición se leen del `.tt.gz`, pero no sustituyen a un registro de protocolo por sujeto.
4. **Etiqueta binaria:** los pacientes agrupan distintos tipos y lateralidades de epilepsia; un único contraste sano-enfermo puede esconder subgrupos (p. ej. foco izquierdo frente a derecho).
5. **Tractos derivados de Autotrack:** los nombres de los clústeres se toman de los archivos de cada sujeto y, cuando existe `tract_stat`, se verifican con los conteos; la identificación anatómica depende del atlas y tolera errores (en el Archivo 2, el arcuato derecho y el ILF derecho de S1 son casi indistinguibles por longitud, 0.03 mm).
6. **Pruebas múltiples:** se reportan valores corregidos (Holm, BH); los p sin corregir son solo exploratorios.
7. **Métricas geométricas dependen del tractograma:** el conteo de streamlines varía con la calidad de la adquisición y la tractografía; por eso se usa también el conteo relativo y se evalúa un modelo de referencia no anatómico.

## **8. Referencias**

* Cliff, N. (1993). Dominance statistics: Ordinal analyses to answer ordinal questions. *Psychological Bulletin*, 114(3), 494–509.
* Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70.
* Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate. *Journal of the Royal Statistical Society B*, 57(1), 289–300.
* Varoquaux, G. (2018). Cross-validation failure: Small sample sizes lead to large error bars. *NeuroImage*, 180, 68–77.
* Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
* Yeh, F.-C. (2020). Shape analysis of the human association pathways. *NeuroImage*, 223, 117329.
* Yeh, F.-C., et al. (2022). Population-averaged atlas of the macroscale human structural connectome and its network topology (Autotrack). *Nature Communications*, 13, 4933.

> Verifique las páginas y autores de cada referencia en PubMed antes de la defensa.